# ML-03 — Frame Your Lane as an ML Task

**Lane:** Refresh / Content Opportunity Scoring  
**Intern:** Varshan M  

This notebook maps the chosen lane onto the ML loop in the order the skeleton specifies:
task type → target → success metric → unit of analysis (as a real dataframe) → why ML beats a rule → self-check.

> Skills loaded per `skills/README.md`: `framing-ml-problems/SKILL.md` + `flyrank/flyrank-data/SKILL.md`

## 1. My lane as an ML task (type)

**Task type: Binary Classification → used to drive a ranked refresh queue (scoring)**

The question the work is trying to answer is: *Which content pages should an editor refresh first?*

That "which ones first?" framing sounds like pure ranking/scoring, but the output has to be *grounded in something real* — an observed outcome, not just a hand-assigned priority.  
The grounding is **classification**: we predict whether a page is currently in a declining-impressions state (`is_declining_label = 1`) or not.  
The predicted probability from that classifier is then used as a **refresh-priority score** — pages with the highest decline probability float to the top of the editor queue.

So the loop is:  
```
Binary classification  →  predicted P(decline)  →  rank by score  →  editor refresh queue
```

This is consistent with the FlyRank pipeline's own design: `scripts/02_baseline_score.py` builds a rule-based refresh score, and `scripts/03_train_model.py` trains a classifier whose predicted probability replaces (or blends with) that score.

**Why not pure ranking/scoring without a label?**  
A scoring approach without an observed target lets you rank pages, but you can never verify whether the ranking was correct — you have no ground truth to measure against. Tying the score to an observed outcome (`is_declining_label`) gives us a real evaluation target.

## 2. Target or proxy

**Target:** `is_declining_label` (binary, 0/1)

**Where it comes from:** This is an *observed outcome*, not an analyst-invented category.  
The pipeline computes it from the data itself:
```
trend_direction = 'down'  ←  impressions_last_30d fell >20% vs impressions_prev_30d
is_declining_label = 1 when trend_direction == 'down'
```
The decline is measured from two non-overlapping 30-day windows of *actual impression counts* from Google Search Console — not from a subjective editorial opinion. The label therefore reflects real search-performance movement.

**The label trap (and why we avoid it):**  
Because `is_declining_label` is derived from `trend_direction` and `trend_pct`, those two columns are **strictly forbidden as model features** — using them would mean the model "predicts" the label using the very rule that defines it (perfect leakage, zero generalization). The data dictionary and `scripts/ml_utils.py` both enforce this.

**The proxy risk:**  
Impression-based decline is a proxy for "page needs refreshing", not a direct measure of content quality. A page's impressions can drop because of algorithm changes, seasonality, or competitor moves — not just because the content is stale. This is worth noting honestly: the model flags *measurably declining pages*, and editors apply judgment about whether a refresh is the right action.

## 3. Success metric

**Primary metric: Precision@K (K = 50)**

**Why Precision@K and not accuracy or ROC-AUC?**

Editors work from a queue. They can realistically review a fixed number of pages per sprint — say the top 50. The model's value is determined by how many of those 50 pages genuinely needed attention (true positives at the top), not by how it performs across all 30,000 rows.  

- **Accuracy** is misleading here: with ~54% of pages labeled as declining, a model that labels everything 1 gets 54% accuracy with zero useful signal.
- **ROC-AUC** is useful for model selection (it's threshold-free), but it doesn't reflect the operational question: "are the pages we're actually sending to editors the right ones?"
- **Precision@50** = (true declining pages in top-50 recommendations) / 50 — directly answers the editor's question.

**What counts as good:**  
The FlyRank reference pipeline's rule-based baseline achieves **Precision@50 = 0.240** (from `outputs/model_report.md`). The random-forest model achieves **~0.740**, roughly a 3x lift over the baseline. For this lane's own model to be useful, it should beat the rule baseline on Precision@50. Beating 0.240 is the minimum bar; matching ~0.740 would be meaningful improvement.

**Secondary metric: ROC-AUC** — for model comparison and threshold selection.

**One metric, defined before training:** Precision@50, measured on a client-holdout test set.

## 4. The unit of analysis, as a real dataframe

**One row = one content page (article) as observed over a trailing 90-day window, for one client.**

The code below loads the starter dataset and shows this grain concretely, along with a sketch of what the target column looks like.

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Resolve the repo root regardless of where the notebook is run from
# Works locally (d:/Flyrank-ML-Internship) and in Colab after git-clone
REPO_CANDIDATES = [
    os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..'),  # local: work/notebooks/
    '/content/Flyrank-ML-Internship',  # Colab after git clone
    os.path.expanduser('~/Flyrank-ML-Internship'),
]

RAW_PATH = None
for candidate in REPO_CANDIDATES:
    p = os.path.normpath(os.path.join(candidate, 'data', 'raw', 'content_refresh_anonymized.csv'))
    if os.path.exists(p):
        RAW_PATH = p
        break

if RAW_PATH is None:
    # Last-resort: search cwd tree
    for root, dirs, files in os.walk('.'):
        if 'content_refresh_anonymized.csv' in files:
            RAW_PATH = os.path.join(root, 'content_refresh_anonymized.csv')
            break

if RAW_PATH is None:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv not found. "
        "In Colab: run `!git clone https://github.com/Varshan-M/Flyrank-ML-Internship.git` first."
    )

print(f"Loading data from: {RAW_PATH}")
df = pd.read_csv(RAW_PATH)
print(f"Shape: {df.shape}  ({df.shape[0]:,} rows x {df.shape[1]} columns)")

Loading data from: D:\Flyrank-ML-Internship\data\raw\content_refresh_anonymized.csv


Shape: (30000, 44)  (30,000 rows x 44 columns)


In [2]:
# Grain check: confirm one row = one content page
print("=== Grain check ===")
print(f"Total rows:                     {len(df):>8,}")
print(f"Unique content_id values:       {df['content_id'].nunique():>8,}")
print(f"Unique client_id values:        {df['client_id'].nunique():>8,}")
print(f"Rows per content_id (should=1): {len(df) / df['content_id'].nunique():.1f}")
print()

# Show what a single row looks like
SHOW_COLS = [
    'content_id', 'client_id', 'content_type', 'content_age_days',
    'impressions_90d', 'clicks_90d', 'ctr', 'avg_position',
    'impressions_last_30d', 'impressions_prev_30d',
    'trend_direction',   # label source -- never a feature
]

print("=== Sample rows (5) -- one row = one content page ===")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
df[SHOW_COLS].head(5)

=== Grain check ===
Total rows:                       30,000
Unique content_id values:         30,000
Unique client_id values:              32
Rows per content_id (should=1): 1.0

=== Sample rows (5) -- one row = one content page ===


,content_id,client_id,content_type,content_age_days,impressions_90d,clicks_90d,ctr,avg_position,impressions_last_30d,impressions_prev_30d,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,187,3803,29,0.76,10.6,578,987,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,15320,7,0.05,20.3,2501,5915,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,12581,11,0.09,36.5,2382,6089,down
3,content_331d6c4de07b,client_19581e27de,keyword article,463,11751,58,0.49,6.2,3626,4206,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,19140,24,0.13,44.0,4211,6452,down


In [3]:
# Sketch the target column
# is_declining_label is added by scripts/01_prepare_features.py;
# we reconstruct it here from raw columns to make the derivation visible.

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("=== Target column: is_declining_label ===")
vc = df['is_declining_label'].value_counts().sort_index()
for val, count in vc.items():
    label_text = 'declining (label=1)' if val == 1 else 'not declining (label=0)'
    print(f"  {val}  {label_text:25s}  {count:6,}  ({count/len(df)*100:.1f}%)")

print()
print("=== trend_direction distribution (the source column) ===")
print(df['trend_direction'].value_counts().to_string())
print()
print("NOTE: 'trend_direction' and 'trend_pct' are label sources -- never model features.")

=== Target column: is_declining_label ===
  0  not declining (label=0)    13,738  (45.8%)
  1  declining (label=1)        16,262  (54.2%)

=== trend_direction distribution (the source column) ===
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

NOTE: 'trend_direction' and 'trend_pct' are label sources -- never model features.


In [4]:
# Unit of analysis with target column
DISPLAY_COLS = [
    'content_id', 'client_id', 'content_type',
    'impressions_90d', 'ctr', 'avg_position',
    'days_since_last_update', 'content_age_days',
    'impressions_last_30d', 'impressions_prev_30d',
    'is_declining_label',   # the target we will predict
]

print("=== Unit of analysis with target column (5 sampled rows) ===")
print("One row = one content page observed over a trailing 90-day window")
print()
df[DISPLAY_COLS].sample(5, random_state=42)

=== Unit of analysis with target column (5 sampled rows) ===
One row = one content page observed over a trailing 90-day window



,content_id,client_id,content_type,impressions_90d,ctr,avg_position,days_since_last_update,content_age_days,impressions_last_30d,impressions_prev_30d,is_declining_label
2308,content_9824710082d8,client_3fdba35f04,keyword article,283,0.00,21.3,104,174,54,50,0
22404,content_3efa3a7c46bb,client_f74efabef1,keyword article,8878,0.09,8.8,20,134,3897,3172,0
23397,content_575dc8a2ab0f,client_25fc0e7096,feedly article,3,0.00,0.0,8,109,3,0,0
25058,content_0dbd6911ba04,client_d029fa3a95,comparison article,124,0.00,8.1,20,151,26,61,1
2664,content_bbaf87019afb,client_19581e27de,keyword article,4294,0.23,30.9,22,466,1157,1461,1


In [5]:
# Declining pages by content_type and freshness_tier
print("=== Declining pages by content_type ===")
breakdown = (
    df.groupby('content_type')['is_declining_label']
    .agg(total='count', declining='sum')
    .assign(decline_rate_pct=lambda x: (x['declining'] / x['total'] * 100).round(1))
    .sort_values('total', ascending=False)
)
print(breakdown.to_string())

print()
print("=== Declining pages by freshness_tier ===")
fresh_breakdown = (
    df.groupby('freshness_tier')['is_declining_label']
    .agg(total='count', declining='sum')
    .assign(decline_rate_pct=lambda x: (x['declining'] / x['total'] * 100).round(1))
    .sort_values('decline_rate_pct', ascending=False)
)
print(fresh_breakdown.to_string())

=== Declining pages by content_type ===
                    total  declining  decline_rate_pct
content_type                                          
keyword article     27207      15262              56.1
feedly article       2096        601              28.7
comparison article    697        399              57.2

=== Declining pages by freshness_tier ===
                total  declining  decline_rate_pct
freshness_tier                                    
91-180           9171       5604              61.1
31-90             175        103              58.9
0-30            20480      10473              51.1
181+              174         82              47.1


## 5. Why ML beats a fixed rule here

**The obvious rule attempt:** "Flag any page where `impressions_last_30d` fell by more than 20% vs `impressions_prev_30d`."  
That IS, literally, the `is_declining_label` definition — so it perfectly identifies declining pages but tells editors *nothing* about which declining pages are worth fixing, and it can't be used as a feature (that would be leakage).

**The harder question (where ML earns its place):** Given only a page's *current* properties and history — without using the trend columns — can we predict which pages are in a declining state, and can we rank-order them better than a simple hand-rule?

A fixed rule can capture one signal at a time. For example:
- "High age + low CTR + low position = needs refresh." — but what counts as high age? It varies by content_type (comparison articles age faster than evergreen keyword articles).
- "If scroll_rate is low and engagement_rate is low, flag it." — but scroll_rate and ai_traffic_pct can exceed 100 (measurement quirks), making clean thresholds unreliable.
- "Pages with avg_position > 20 need refresh." — but a page sitting at position 30 with rising impressions is likely recovering, not declining.

**Why the pattern is too messy for if-statements:**

1. **Many signals interact.** Whether a page is declining depends on the combination of position, CTR, session trend, content age, freshness, keyword competition, and content type — not any single threshold. A decision tree can learn that *old + low-position + no-recent-update + keyword-article* is a strong predictor, while the same age is fine for a comparison article with high engagement.

2. **Missingness is systematic, not random.** Keyword columns are missing for `feedly article` rows entirely (~100%); word_count is missing for ~25% of rows. A hand-rule must explicitly handle every case; a model with `has_`-flag features can learn missingness as a signal.

3. **The class balance shifts.** ~54% of pages are declining in this snapshot. A fixed rule tuned on this window will mis-calibrate as the proportion shifts with algorithm updates. A model can be retrained; a rule requires manual revision.

4. **The baseline shows the gap concretely.** The reference pipeline's rule-based baseline achieves **Precision@50 = 0.240** (from `outputs/model_report.md`). The random-forest model reaches **~0.740** — roughly 3x better. That gap is the direct cost of using only a rule.

**What the output supports:**  
The classifier produces a probability score for each page. An editor or content manager uses the ranked queue to decide which pages to open and review. The model does not automatically update content — it surfaces candidates. The human makes the final call on whether to refresh, rewrite, or retire each page.

**One-paragraph frame (from the skill template):**

> For content editors, deciding which pages to refresh first, we will build a binary classifier from 90-day search and engagement signals, predicting `is_declining_label` (1 = impressions fell >20% last 30 days vs prior 30 days), scored by **Precision@50** on a client-holdout test set. A wrong call costs wasted editor time on a stable page, or a missed chance on a genuinely declining one. A plain if-statement isn't enough because decline depends on a complex, content-type-sensitive interaction of position, CTR, age, freshness, and engagement signals that shift over time. We will claim only directional, decision-support results — not that refreshing the flagged pages will *cause* rankings to improve.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.